In [3]:
import pandas as pd

df = pd.read_csv("../data/sample_report.csv")

df

,Site,Device,Month,Consumption_kWh,COP,Alarms
0,Helsinki,Pump A,2026-05,1200,3.5,2
1,Helsinki,Pump A,2026-06,1450,3.1,4
2,Espoo,Pump B,2026-05,950,3.8,1
3,Espoo,Pump B,2026-06,980,3.7,1
4,Vantaa,Pump C,2026-05,2200,3.2,3
5,Vantaa,Pump C,2026-06,2700,2.8,6
6,Turku,Pump D,2026-05,1500,3.6,0
7,Turku,Pump D,2026-06,1520,3.5,1


In [4]:
may = df[df["Month"] == "2026-05"]
print(may)

       Site  Device    Month  Consumption_kWh  COP  Alarms
0  Helsinki  Pump A  2026-05             1200  3.5       2
2     Espoo  Pump B  2026-05              950  3.8       1
4    Vantaa  Pump C  2026-05             2200  3.2       3
6     Turku  Pump D  2026-05             1500  3.6       0


In [5]:
df.groupby("Month")["Consumption_kWh"].sum()

Month
2026-05    5850
2026-06    6650
Name: Consumption_kWh, dtype: int64

In [6]:
may = df[df["Month"] == "2026-05"]
june = df[df["Month"] == "2026-06"]

comparison = may.merge(
    june,
    on="Device",
    suffixes=("_may", "_june")
    )

delta_cop=comparison["cop_change"] = (
        comparison["COP_june"]
        - comparison["COP_may"]
    )
delta_alarms=comparison["alarms_diff"] = (
        comparison["Alarms_june"]
        - comparison["Alarms_may"]
    )
comparison[abs(delta_cop) > 1]
comparison[(delta_alarms) > 5]



,Site_may,Device,Month_may,Consumption_kWh_may,COP_may,Alarms_may,Site_june,Month_june,Consumption_kWh_june,COP_june,Alarms_june,cop_change,alarms_diff


In [7]:
cop_anomalies = comparison[abs(delta_cop) > 0.3]
cop_anomalies

,Site_may,Device,Month_may,Consumption_kWh_may,COP_may,Alarms_may,Site_june,Month_june,Consumption_kWh_june,COP_june,Alarms_june,cop_change,alarms_diff
0,Helsinki,Pump A,2026-05,1200,3.5,2,Helsinki,2026-06,1450,3.1,4,-0.4,2
2,Vantaa,Pump C,2026-05,2200,3.2,3,Vantaa,2026-06,2700,2.8,6,-0.4,3


In [8]:
cop_anomalies["Device"][0]

'Pump A'

In [9]:
comparison[["Device", "COP_may", "COP_june", "cop_change", "Alarms_may", "Alarms_june", "alarms_diff"]]

,Device,COP_may,COP_june,cop_change,Alarms_may,Alarms_june,alarms_diff
0,Pump A,3.5,3.1,-0.4,2,4,2
1,Pump B,3.8,3.7,-0.1,1,1,0
2,Pump C,3.2,2.8,-0.4,3,6,3
3,Pump D,3.6,3.5,-0.1,0,1,1


In [10]:
comparison[[
    "Device",
    "Alarms_may",
    "Alarms_june",
    "alarms_diff"
]]

,Device,Alarms_may,Alarms_june,alarms_diff
0,Pump A,2,4,2
1,Pump B,1,1,0
2,Pump C,3,6,3
3,Pump D,0,1,1


In [11]:
df["Month"].unique()


array(['2026-05', '2026-06'], dtype=object)

In [12]:
sorted(df["Month"].unique())

['2026-05', '2026-06']

In [13]:
months = sorted(df["Month"].unique())

In [14]:
months[-1]

'2026-06'

In [15]:
previous_month = months[-2]
latest_month = months[-1]

previous_month, latest_month

('2026-05', '2026-06')

In [16]:
previous = df[df["Month"] == previous_month]
latest = df[df["Month"] == latest_month]

In [17]:
previous

,Site,Device,Month,Consumption_kWh,COP,Alarms
0,Helsinki,Pump A,2026-05,1200,3.5,2
2,Espoo,Pump B,2026-05,950,3.8,1
4,Vantaa,Pump C,2026-05,2200,3.2,3
6,Turku,Pump D,2026-05,1500,3.6,0


In [18]:
latest

,Site,Device,Month,Consumption_kWh,COP,Alarms
1,Helsinki,Pump A,2026-06,1450,3.1,4
3,Espoo,Pump B,2026-06,980,3.7,1
5,Vantaa,Pump C,2026-06,2700,2.8,6
7,Turku,Pump D,2026-06,1520,3.5,1


In [20]:
comparison = previous.merge(
    latest,
    on="Device",
    suffixes=("_previous", "_latest")
)

In [21]:
comparison

,Site_previous,Device,Month_previous,Consumption_kWh_previous,COP_previous,Alarms_previous,Site_latest,Month_latest,Consumption_kWh_latest,COP_latest,Alarms_latest
0,Helsinki,Pump A,2026-05,1200,3.5,2,Helsinki,2026-06,1450,3.1,4
1,Espoo,Pump B,2026-05,950,3.8,1,Espoo,2026-06,980,3.7,1
2,Vantaa,Pump C,2026-05,2200,3.2,3,Vantaa,2026-06,2700,2.8,6
3,Turku,Pump D,2026-05,1500,3.6,0,Turku,2026-06,1520,3.5,1


In [22]:
comparison["COP_latest"] - comparison["COP_previous"]

0   -0.4
1   -0.1
2   -0.4
3   -0.1
dtype: float64

In [23]:
comparison["cop_change"] = (
    comparison["COP_latest"]
    - comparison["COP_previous"]
)

In [24]:
comparison["alarms_diff"] = (
    comparison["Alarms_latest"]
    - comparison["Alarms_previous"]
)

In [25]:
comparison[
    ["Device", "COP_previous", "COP_latest",
     "cop_change", "Alarms_previous",
     "Alarms_latest", "alarms_diff"]
]

,Device,COP_previous,COP_latest,cop_change,Alarms_previous,Alarms_latest,alarms_diff
0,Pump A,3.5,3.1,-0.4,2,4,2
1,Pump B,3.8,3.7,-0.1,1,1,0
2,Pump C,3.2,2.8,-0.4,3,6,3
3,Pump D,3.6,3.5,-0.1,0,1,1
